# ⚡ GPU-Accelerated Fraud Detection
### CPU vs GPU inference benchmark on 6M+ real financial transactions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adibis-git/fraud-detection-gpu-demo/blob/main/fraud_detection_gpu_demo.ipynb)

**What this notebook shows:**
- Load a real financial transaction dataset (PaySim, 6.36M rows)
- Train an XGBoost fraud detection model
- Benchmark inference speed: **CPU vs GPU**
- Visualise the throughput gap — the number that matters in BFSI real-time fraud scoring

**Runtime:** ~10 minutes on Colab free tier (T4 GPU)  
**Prerequisite:** Runtime → Change runtime type → **T4 GPU** ← do this first

---

## 1. Environment Setup

In [ ]:
import subprocess, sys

# Install dependencies quietly
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "xgboost>=2.0", "datasets", "matplotlib", "pandas", "scikit-learn"],
    check=True
)

print("✅ Dependencies installed")

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU detected: {result.stdout.strip()}")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU, then re-run.")

## 2. Load the PaySim Financial Transaction Dataset

PaySim simulates mobile money transactions based on aggregated data from a real BFSI deployment.  
**6.36 million transactions · 11 features · No login required**

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Loading PaySim dataset from Hugging Face (this takes ~60s on first run)...")
raw = load_dataset(
    "purulalwani/Synthetic-Financial-Datasets-For-Fraud-Detection",
    split="train"
)
df = raw.to_pandas()
print(f"✅ Loaded {len(df):,} transactions")
df.head(3)

In [ ]:
print("Dataset overview")
print("-" * 40)
print(f"Rows          : {len(df):,}")
print(f"Fraud cases   : {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.2f}%)")
print(f"\nTransaction types:")
print(df['type'].value_counts().to_string())
print(f"\nAmount range  : ${df['amount'].min():,.2f} – ${df['amount'].max():,.2f}")

## 3. Feature Engineering & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Encode transaction type
le = LabelEncoder()
df['type_enc'] = le.fit_transform(df['type'])

# Balance-delta features — key signal in fraud detection
df['orig_balance_delta'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['dest_balance_delta'] = df['newbalanceDest'] - df['oldbalanceDest']
df['amount_to_orig_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
df['zero_orig_after'] = (df['newbalanceOrig'] == 0).astype(int)

FEATURES = [
    'type_enc', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'orig_balance_delta', 'dest_balance_delta',
    'amount_to_orig_ratio', 'zero_orig_after'
]
TARGET = 'isFraud'

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set : {len(X_train):,} rows")
print(f"Test set  : {len(X_test):,} rows  ← used for inference benchmark")

## 4. Train XGBoost Fraud Model (on GPU)

XGBoost natively supports CUDA-accelerated training and inference.  
We train once on GPU, then benchmark **prediction** speed on CPU vs GPU.

In [ ]:
import xgboost as xgb
import time

print(f"XGBoost version: {xgb.__version__}")

# DMatrix is XGBoost's optimised data structure
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params_gpu = {
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'auc',
    'device'           : 'cuda',          # GPU training
    'max_depth'        : 6,
    'learning_rate'    : 0.1,
    'n_estimators'     : 300,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'scale_pos_weight' : int((y_train == 0).sum() / (y_train == 1).sum()),
    'seed'             : 42,
    'verbosity'        : 0,
}

t0 = time.perf_counter()
model_gpu = xgb.train(
    params_gpu, dtrain,
    num_boost_round=300,
    evals=[(dtest, 'test')],
    verbose_eval=50
)
train_time = time.perf_counter() - t0
print(f"\n✅ GPU training complete in {train_time:.1f}s")

## 5. Model Quality Check

In [ ]:
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix
)

preds_proba = model_gpu.predict(dtest)
preds_binary = (preds_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, preds_proba)
print(f"AUC-ROC : {auc:.4f}")
print()
print(classification_report(y_test, preds_binary, target_names=['Legit', 'Fraud']))

## 6. ⚡ The Benchmark: CPU vs GPU Inference

This is the core of the demo.  
Same model, same 1.27M test transactions — only the **inference device** changes.

> **Why this matters in BFSI:** A real-time fraud engine must score each transaction *before it clears* — typically a 200ms window. At high transaction volumes (Black Friday, IPO day), CPU-based inference becomes the bottleneck.

In [ ]:
import time
import numpy as np

N_WARMUP = 3
N_RUNS   = 5

def benchmark_inference(model, dmatrix, device_label, n_runs=N_RUNS):
    """Run inference N times and return mean latency and throughput."""
    n_rows = dmatrix.num_row()
    latencies = []

    # Warm-up
    for _ in range(N_WARMUP):
        _ = model.predict(dmatrix)

    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = model.predict(dmatrix)
        latencies.append(time.perf_counter() - t0)

    mean_latency_ms = np.mean(latencies) * 1000
    throughput_tps  = n_rows / np.mean(latencies)
    return mean_latency_ms, throughput_tps


# --- CPU benchmark ---
print("Running CPU inference benchmark...")
params_cpu = dict(params_gpu)
params_cpu['device'] = 'cpu'

model_cpu = xgb.train(
    params_cpu, dtrain,
    num_boost_round=300,
    verbose_eval=False
)
dtest_cpu = xgb.DMatrix(X_test)
cpu_latency_ms, cpu_tps = benchmark_inference(model_cpu, dtest_cpu, "CPU")
print(f"  CPU  → {cpu_latency_ms:>8.1f} ms  |  {cpu_tps:>12,.0f} tx/sec")


# --- GPU benchmark ---
print("\nRunning GPU inference benchmark...")
dtest_gpu = xgb.DMatrix(X_test)
gpu_latency_ms, gpu_tps = benchmark_inference(model_gpu, dtest_gpu, "GPU")
print(f"  GPU  → {gpu_latency_ms:>8.1f} ms  |  {gpu_tps:>12,.0f} tx/sec")


speedup = gpu_tps / cpu_tps
latency_reduction = (1 - gpu_latency_ms / cpu_latency_ms) * 100

print(f"\n{'='*50}")
print(f"  GPU is {speedup:.1f}× faster in throughput")
print(f"  Latency reduced by {latency_reduction:.0f}%")
print(f"{'='*50}")

## 7. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("CPU vs GPU · XGBoost Fraud Inference\nPaySim Dataset · 1.27M Transactions",
             fontsize=14, fontweight='bold', y=1.02)

COLORS = ['#d9534f', '#2ecc71']   # red = CPU, green = GPU
LABELS = ['CPU', 'GPU']

# --- Throughput ---
ax = axes[0]
bars = ax.bar(LABELS, [cpu_tps, gpu_tps], color=COLORS, width=0.45,
              edgecolor='white', linewidth=1.5)
ax.set_title('Throughput (transactions / second)', fontweight='bold')
ax.set_ylabel('tx / sec')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
ax.set_ylim(0, gpu_tps * 1.25)
ax.spines[['top', 'right']].set_visible(False)

for bar, val in zip(bars, [cpu_tps, gpu_tps]):
    label = f"{val/1e6:.2f}M" if val >= 1e6 else f"{val/1e3:.0f}K"
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + gpu_tps*0.02,
            label, ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.annotate(f"{speedup:.1f}× faster",
            xy=(1, gpu_tps), xytext=(0.5, gpu_tps * 1.15),
            fontsize=13, color='#2ecc71', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2))

# --- Latency ---
ax2 = axes[1]
bars2 = ax2.bar(LABELS, [cpu_latency_ms, gpu_latency_ms], color=COLORS, width=0.45,
                edgecolor='white', linewidth=1.5)
ax2.set_title('Inference Latency (full test set, ms)', fontweight='bold')
ax2.set_ylabel('milliseconds')
ax2.set_ylim(0, cpu_latency_ms * 1.3)
ax2.spines[['top', 'right']].set_visible(False)

for bar, val in zip(bars2, [cpu_latency_ms, gpu_latency_ms]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + cpu_latency_ms*0.02,
             f"{val:.0f} ms", ha='center', va='bottom', fontweight='bold', fontsize=12)

ax2.annotate(f"{latency_reduction:.0f}% reduction",
             xy=(1, gpu_latency_ms), xytext=(0.4, cpu_latency_ms * 0.7),
             fontsize=13, color='#2ecc71', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2))

plt.tight_layout()
plt.savefig('cpu_vs_gpu_fraud_inference.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as cpu_vs_gpu_fraud_inference.png")

## 8. Results Summary

In [ ]:
# Cost estimate: rough AWS on-demand pricing (us-east-1, Jun 2025)
cpu_price_per_hr  = 0.096   # c5.2xlarge (8 vCPU)
gpu_price_per_hr  = 0.526   # g4dn.xlarge (T4)

cost_cpu_per_1M = (1_000_000 / cpu_tps) / 3600 * cpu_price_per_hr
cost_gpu_per_1M = (1_000_000 / gpu_tps) / 3600 * gpu_price_per_hr

results = {
    'Metric'                        : ['Throughput (tx/sec)', 'Latency (ms, 1.27M tx)', 'Cost per 1M inferences (USD)'],
    'CPU (c5.2xlarge)'              : [f"{cpu_tps:,.0f}", f"{cpu_latency_ms:.0f} ms", f"${cost_cpu_per_1M:.4f}"],
    'GPU (g4dn.xlarge / T4)'        : [f"{gpu_tps:,.0f}", f"{gpu_latency_ms:.0f} ms", f"${cost_gpu_per_1M:.4f}"],
    'GPU Advantage'                 : [f"{speedup:.1f}× faster", f"{latency_reduction:.0f}% lower",
                                       f"{cost_cpu_per_1M/cost_gpu_per_1M:.1f}× cheaper per inference"]
}

summary_df = pd.DataFrame(results)
summary_df.set_index('Metric', inplace=True)
print(summary_df.to_string())

---

## What this means for your fraud stack

| Scenario | CPU | GPU |
|---|---|---|
| Peak transaction burst (e.g. festive sale) | Latency spikes, queue builds | Absorbs burst within SLA |
| Real-time card authorisation (< 200 ms) | Risk of timeout at scale | Comfortable headroom |
| Model retraining cycle | Slower iteration | Faster experimentation |
| Cost at 100M inferences / day | Higher compute bill | Lower cost per inference |

**GPU inference is not just faster — it's more cost-effective per transaction at scale.**

---

### Want to see how this applies to your architecture?

This benchmark is a starting point. Real BFSI deployments layer on:
- Model explainability (SHAP) for regulatory compliance
- Online feature stores for sub-10ms feature retrieval
- Model monitoring and drift detection in production

**Reach out on LinkedIn if you're evaluating GPU inference for your fraud or risk platform.**

---
*Notebook built with XGBoost 2.x + PaySim dataset · Runs on Colab free tier (T4 GPU)*